# LoRA 微調整エンドツーエンド Notebook

本ノートブックは以下を単一環境で再現します:
1. 環境セットアップ / 依存インストール
2. データロード (CSV / JSONL) と前処理
3. モデル & トークナイザ読み込み + 4bit/FP16 自動判定
4. LoRA (または QLoRA 相当) 微調整
5. 学習ログ収集 & 可視化
6. 推論テスト / ベンチマーク (速度, VRAM, トークン統計)
7. Perplexity / 追加メトリクス計算
8. LoRA マージ & 配布用保存
9. chat インタラクション簡易UI (TextStreamer)
10. レポート集約とバージョニング

---
実行順序: 上から順に。途中でパラメータを変更した場合は *設定パラメータ定義* 以降を再実行してください。


In [ ]:
# 1. 環境設定とライブラリインストール
%pip install -q -U transformers datasets peft accelerate bitsandbytes pynvml evaluate pandas numpy matplotlib

import importlib, sys, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"Torch: {torch.__version__}")
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import issue:", e)


In [ ]:
# 2. Hugging Face ログイン & パス初期化
from huggingface_hub import login
import os, pathlib, getpass, datetime

# 環境変数経由のトークンがあれば入力不要
token_env = os.getenv("HUGGINGFACE_TOKEN")
if token_env:
    login(token=token_env)
else:
    try:
        login()  # UI で入力
    except Exception as e:
        print("[WARN] login skipped:", e)

# ルートディレクトリ設定 (Colab / ローカル両対応)
ROOT = pathlib.Path(os.getenv("WORK_DIR", ".")).resolve()
RUNS_ROOT = ROOT / "results"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
MODELS_ROOT = ROOT / "models"
MODELS_ROOT.mkdir(exist_ok=True)
LOG_ROOT = ROOT / "logs"
LOG_ROOT.mkdir(exist_ok=True)
print("ROOT=", ROOT)


In [ ]:
# 3. 乱数シード & GPU/VRAM ユーティリティ
import random, numpy as np, torch, json
try:
    import pynvml
    pynvml.nvmlInit()
except Exception:
    pynvml = None

SEED = 42

def set_seed(seed: int = 42):
    import os
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[seed fixed] {seed}")

set_seed(SEED)

def vram_report():
    if not torch.cuda.is_available():
        return {"cuda": False}
    rep = {"cuda": True}
    rep["gpu_name"] = torch.cuda.get_device_name(0)
    try:
        if pynvml:
            h = pynvml.nvmlDeviceGetHandleByIndex(0)
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            rep.update({
                "total_gb": round(mem.total/1024**3,2),
                "free_gb": round(mem.free/1024**3,2),
                "used_gb": round(mem.used/1024**3,2),
            })
    except Exception:
        pass
    print(rep)
    return rep

_ = vram_report()


In [ ]:
# 4. 設定パラメータ定義 (Parameter Block)
from dataclasses import dataclass, asdict

# モデル / 学習 / 生成 / 経路 パラメータ (大文字=hash対象)
MODEL_ID = "Qwen/Qwen2-7B-Instruct"  # 例: Qwen/Qwen2-7B-Instruct
MODE = "auto"            # "4bit" | "fp16" | "auto"
MAX_SEQ_LEN = 2048
MAX_NEW_TOKENS = 256
LR = 2e-4
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACC = 8
WARMUP_RATIO = 0.03
EVAL_STEPS = 100
SAVE_STEPS = 200
LOGGING_STEPS = 10
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
USE_QLORA = True  # 4bit での学習前提
PROMPT_SAMPLE = "あなたは丁寧な日本語アシスタントです。次の質問に簡潔に答えてください。\n質問: LORAとは何ですか?\n回答:"
VAL_SPLIT = 0.05
SEED = 42
BENCH_PROMPTS = [
    "Qwenはどのようなアーキテクチャですか? 50文字以内で要約してください。",
    "PythonでFizzBuzzを書くサンプルコードをください。",
]
OUTPUT_BASE = RUNS_ROOT  # ルート
RUN_ID = datetime.datetime.utcnow().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_BASE / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR=", RUN_DIR)


In [ ]:
# 4a. パス手動設定 (環境変数を使わない場合の明示指定)
# 必要に応じて以下を書き換えて実行してください。
# 例: Google Drive を /content/drive/MyDrive/llm-lab-save にマウントした後
# DATA_ROOT = Path('/content/drive/MyDrive/llm-lab-save')
from pathlib import Path

DATA_ROOT = Path('/content/llm-lab-save/')  # ここを変更
DATASETS_DIR = DATA_ROOT / 'datasets'
MODELS_DIR = DATA_ROOT / 'models'
RESULTS_DIR = DATA_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
DATASETS_DIR.mkdir(exist_ok=True, parents=True)

# 個別ファイルを直接指定したい場合 (指定しなければ後段でデモデータ使用)
CSV_PATH = ''  # 例: str(DATASETS_DIR / 'train.csv')
JSONL_PATH = ''  # 例: str(DATASETS_DIR / 'train.jsonl')

print('[PATHS]', {'DATA_ROOT': str(DATA_ROOT), 'DATASETS_DIR': str(DATASETS_DIR)})


In [ ]:
# 5. データセット読み込み (CSV / JSONL)
import pandas as pd
from datasets import Dataset, load_dataset

# 4a セルで手動設定された CSV_PATH / JSONL_PATH を優先
# 空の場合のみ環境変数を参照 (後方互換)
if not CSV_PATH:
    CSV_PATH = os.getenv("TRAIN_CSV", "")
if not JSONL_PATH:
    JSONL_PATH = os.getenv("TRAIN_JSONL", "")

raw_records = []
source_type = None

if CSV_PATH and os.path.exists(CSV_PATH):
    df_csv = pd.read_csv(CSV_PATH)
    print("[LOAD] CSV rows=", len(df_csv))
    raw_records = df_csv.to_dict("records")
    source_type = "csv"
elif JSONL_PATH and os.path.exists(JSONL_PATH):
    ds_json = load_dataset("json", data_files=JSONL_PATH, split="train")
    print("[LOAD] JSONL rows=", len(ds_json))
    raw_records = list(ds_json)
    source_type = "jsonl"
else:
    # デモ用サンプル (小規模)
    raw_records = [
        {"prompt": "こんにちは。自己紹介して", "completion": "私はサンプルAIです。よろしくお願いします。"},
        {"prompt": "LoRAとは?", "completion": "低ランク近似で効率的に微調整する手法です。"},
        {"prompt": "Pythonで足し算", "completion": "print(1+2) # 3"},
        {"prompt": "GPUとは?", "completion": "並列計算に特化したプロセッサです。"},
        {"prompt": "Transformerの要素", "completion": "Self-Attention, FFN, Residual, LayerNorm などです。"},
    ]
    print("[INFO] Fallback demo dataset size=", len(raw_records))
    source_type = "demo"

print("source_type=", source_type)


In [ ]:
# 6. データ検査 & クリーニング
import math, statistics
import numpy as np

cleaned = []
for r in raw_records:
    # 既存 messages 形式の場合は後段で再利用 (ここでは通過)
    if 'messages' in r:
        cleaned.append(r)
        continue
    p = (r.get('prompt') or '').strip()
    c = (r.get('completion') or '').strip()
    if not p or not c:
        continue
    if len(p) < 2 or len(c) < 2:
        continue
    if len(p) > 8000 or len(c) > 8000:
        continue
    cleaned.append({'prompt': p, 'completion': c})

print(f"[CLEAN] before={len(raw_records)} after={len(cleaned)}")

lengths_prompt = [len(x.get('prompt','')) for x in cleaned if 'prompt' in x]
if lengths_prompt:
    print("prompt chars mean/median/max=", np.mean(lengths_prompt), statistics.median(lengths_prompt), max(lengths_prompt))


In [ ]:
# 7. 会話形式への整形 (messages 形式生成)
from typing import List, Dict

if cleaned and 'messages' not in cleaned[0]:
    def format_row(r: Dict) -> Dict:
        return {
            'messages': [
                {'role': 'user', 'content': r['prompt']},
                {'role': 'assistant', 'content': r['completion']}
            ]
        }
    records_msgs = [format_row(r) for r in cleaned]
else:
    records_msgs = cleaned[:]  # 既に messages 形式

print("[FORMAT] samples=", len(records_msgs))
print(records_msgs[0])


In [ ]:
# 4b. 再現性用 config_hash 生成
import hashlib, inspect, json

def collect_param_block(globals_dict):
    allow_types = (int, float, bool, str, list, dict)
    params = {}
    for k,v in globals_dict.items():
        if k.isupper() and isinstance(v, allow_types):
            params[k] = v
    return params

PARAM_BLOCK = collect_param_block(globals())
_param_json = json.dumps(PARAM_BLOCK, sort_keys=True, ensure_ascii=False, separators=(",",":"))
config_hash = hashlib.sha256(_param_json.encode()).hexdigest()[:16]
print("config_hash=", config_hash)
with open(RUN_DIR/"parameter_snapshot.json","w",encoding="utf-8") as f:
    f.write(_param_json)


In [ ]:
# 8. Train / Validation 分割
from datasets import Dataset
full_ds = Dataset.from_list(records_msgs)
split = full_ds.train_test_split(test_size=VAL_SPLIT, seed=SEED)
train_ds = split['train']
val_ds = split['test']
print(train_ds, val_ds)
print(f"train={len(train_ds)} val={len(val_ds)} ratio={len(val_ds)/(len(train_ds)+len(val_ds)):.3f}")


In [ ]:
# 9. トークナイザ読み込み & Chatテンプレート適用
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(build_prompt(train_ds[0]['messages'])[:200])


In [ ]:
# 10. トークナイズ & 長さ統計
import numpy as np

def tokenize_example(ex):
    prompt = build_prompt(ex['messages'])
    out = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN)
    out['input_len'] = len(out['input_ids'])
    out['labels'] = out['input_ids'].copy()
    return out

train_tok = train_ds.map(tokenize_example, remove_columns=['messages'])
val_tok = val_ds.map(tokenize_example, remove_columns=['messages'])

lengths = train_tok['input_len']
print("len stats mean/median/p95/max=", np.mean(lengths), np.median(lengths), np.percentile(lengths,95), max(lengths))


In [ ]:
# 11. 動的パディング Collator
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
# 12. LoRA 設定 & モデルロード (4bit/FP16 自動)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

use_cuda = torch.cuda.is_available()
# モード決定
chosen_mode = MODE
if MODE == 'auto' and use_cuda:
    free_gb = torch.cuda.mem_get_info()[0]/1024**3
    # 目安: 14GB 以上空いてれば fp16, それ以下は4bit
    chosen_mode = 'fp16' if free_gb >= 14 else '4bit'
print("[mode]", chosen_mode)

bnb_cfg = None
load_kwargs = dict(trust_remote_code=True, attn_implementation='sdpa')
if use_cuda:
    load_kwargs['device_map'] = 'auto'

if chosen_mode == '4bit':
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if use_cuda else torch.float16,
    )
    load_kwargs['quantization_config'] = bnb_cfg
    load_kwargs['torch_dtype'] = torch.bfloat16 if use_cuda else torch.float16
else:
    load_kwargs['torch_dtype'] = torch.bfloat16 if use_cuda else torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
if chosen_mode == '4bit' and USE_QLORA:
    model = prepare_model_for_kbit_training(model)

model.config.use_cache = False
if tokenizer.pad_token is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
# 13. メモリ最適化設定
if use_cuda:
    torch.cuda.empty_cache()
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()


In [ ]:
# 14. TrainingArguments 定義
from transformers import TrainingArguments

OUTPUT_DIR = RUN_DIR / "adapter"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    evaluation_strategy='steps',
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to=["none"],
    fp16=(not bf16_ok and torch.cuda.is_available()),
    bf16=(bf16_ok),
    optim='adamw_8bit',
)
print(training_args)


In [ ]:
# 15. Trainer 初期化 & 学習
from transformers import Trainer
import csv, time, math

LOG_TRAIN_CSV = RUN_DIR/"training_log.csv"
LOG_TRAIN_JSONL = RUN_DIR/"training_log.jsonl"

class SimpleLoggerCallback:
    def __init__(self):
        self.header_written = False
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        row = {
            'step': state.global_step,
            'loss': logs.get('loss'),
            'lr': logs.get('learning_rate'),
            'epoch': state.epoch,
        }
        # CSV
        exists = LOG_TRAIN_CSV.exists()
        with open(LOG_TRAIN_CSV,'a',newline='') as f:
            w=csv.DictWriter(f, fieldnames=row.keys())
            if not exists:
                w.writeheader()
            w.writerow(row)
        # JSONL
        with open(LOG_TRAIN_JSONL,'a',encoding='utf-8') as f:
            f.write(json.dumps(row, ensure_ascii=False)+"\n")

callback_logger = SimpleLoggerCallback()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    tokenizer=tokenizer,
    callbacks=[callback_logger],
)

train_result = trainer.train()
print(train_result)


In [ ]:
# 16. 学習曲線可視化 & 中間保存
import pandas as pd, matplotlib.pyplot as plt
if LOG_TRAIN_CSV.exists():
    df_log = pd.read_csv(LOG_TRAIN_CSV)
    if 'loss' in df_log.columns:
        plt.figure(figsize=(5,3))
        plt.plot(df_log['step'], df_log['loss'])
        plt.xlabel('step'); plt.ylabel('loss'); plt.title('Train Loss')
        plt.tight_layout()
        fig_path = RUN_DIR/"loss_curve.png"
        plt.savefig(fig_path)
        print("saved:", fig_path)
    else:
        print("[WARN] loss column not found")
else:
    print("[WARN] training log not found")

# モデル/トークナイザ保存 (adapter)
trainer.save_model()
print("[SAVE] adapter saved to", OUTPUT_DIR)


In [ ]:
# 17. 学習済み LoRA アダプター推論テスト
from peft import AutoPeftModelForCausalLM

def quick_infer(prompt: str, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors='pt')
    if use_cuda:
        inputs = {k:v.to('cuda') for k,v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample_out = quick_infer(PROMPT_SAMPLE)
print(sample_out)


In [ ]:
# 18. ベンチマーク計測 (TTFT, Throughput, VRAM)
from transformers import TextIteratorStreamer
import time

BENCH_CSV = RUN_DIR/"bench.csv"
bench_header = ["timestamp","config_hash","model_id","mode","prompt_tokens","new_tokens","ttft_ms","gen_ms","total_ms","tok_per_s","peak_mem_gb"]

if use_cuda:
    torch.cuda.reset_peak_memory_stats()

rows = []
for p in BENCH_PROMPTS:
    enc = tokenizer(p, return_tensors='pt')
    if use_cuda:
        enc = {k:v.to('cuda') for k,v in enc.items()}
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, streamer=streamer)
    start = time.time(); first_token_t = None
    tokens_generated = 0

    def _generate():
        with torch.inference_mode():
            model.generate(**gen_kwargs)
    import threading
    th = threading.Thread(target=_generate)
    th.start()
    for chunk in streamer:
        if first_token_t is None:
            first_token_t = time.time()
        tokens_generated += len(tokenizer(chunk).input_ids)
    th.join()
    end = time.time()

    ttft_ms = (first_token_t - start)*1000 if first_token_t else None
    total_ms = (end - start)*1000
    gen_ms = (end - first_token_t)*1000 if first_token_t else None
    tok_per_s = tokens_generated / (end - first_token_t) if first_token_t else None
    peak_gb = torch.cuda.max_memory_allocated()/1024**3 if use_cuda else None

    row = [datetime.datetime.utcnow().isoformat(), config_hash, MODEL_ID, chosen_mode,
           enc['input_ids'].shape[-1], tokens_generated, round(ttft_ms,2) if ttft_ms else '',
           round(gen_ms,2) if gen_ms else '', round(total_ms,2),
           round(tok_per_s,2) if tok_per_s else '', round(peak_gb,3) if peak_gb else '']
    rows.append(row)

# 書き込み
import csv as _csv
exists = BENCH_CSV.exists()
with open(BENCH_CSV,'a',newline='') as f:
    w = _csv.writer(f)
    if not exists:
        w.writerow(bench_header)
    w.writerows(rows)
print("[BENCH] rows appended=", len(rows))


In [ ]:
# 19. Perplexity 評価 (簡易)
import math, torch
from torch.nn import CrossEntropyLoss

def compute_ppl(dataset_tok, max_batches=10):
    model.eval()
    losses = []
    bs = 1
    for i, ex in enumerate(dataset_tok):
        if i >= max_batches: break
        input_ids = torch.tensor([ex['input_ids']], device='cuda' if use_cuda else 'cpu')
        labels = torch.tensor([ex['labels']], device=input_ids.device)
        with torch.inference_mode():
            out = model(input_ids=input_ids, labels=labels)
        losses.append(out.loss.item())
    if not losses:
        return None
    ce = sum(losses)/len(losses)
    return math.exp(ce)

val_ppl = compute_ppl(val_tok, max_batches=min(20, len(val_tok)))
print("val_ppl=", val_ppl)
with open(RUN_DIR/"metrics.json","w",encoding="utf-8") as f:
    json.dump({"val_ppl": val_ppl}, f, ensure_ascii=False, indent=2)


In [ ]:
# 20. LoRA マージ & 保存
MERGED_DIR = RUN_DIR/"merged"
MERGED_DIR.mkdir(exist_ok=True)

try:
    merged = model.merge_and_unload()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print("[MERGE] merged model saved", MERGED_DIR)
except Exception as e:
    print("[WARN] merge failed (continue with adapter):", e)


In [ ]:
# 21. チャットインタラクション (ストリーミング)
from transformers import TextStreamer

history = []
stop_ids = [tokenizer.eos_token_id]
try:
    im_end = tokenizer.convert_tokens_to_ids('<|im_end|>')
    if isinstance(im_end,int) and im_end not in stop_ids and im_end != tokenizer.unk_token_id:
        stop_ids.append(im_end)
except Exception: pass

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def chat_once(user_text: str, max_new_tokens=128):
    history.append({'role':'user','content':user_text})
    prompt = build_prompt(history)
    enc = tokenizer(prompt, return_tensors='pt')
    if use_cuda:
        enc = {k:v.to('cuda') for k,v in enc.items()}
    print('Assistant:', end=' ', flush=True)
    with torch.inference_mode():
        model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer,
                       eos_token_id=stop_ids, pad_token_id=tokenizer.pad_token_id)
    # 後で最後の assistant テキストを再構築 (簡略: 全履歴から再抽出)
    # ここでは履歴保存のみ
    return

# chat_once("こんにちは。自己紹介して。")  # 例


In [ ]:
# 22. 評価レポート集約
import json, os, math, csv
report = {
    'run_id': RUN_ID,
    'config_hash': config_hash,
    'model_id': MODEL_ID,
    'mode': chosen_mode,
    'val_ppl': val_ppl,
}
# LoRA trainable パラメータ割合
if hasattr(model, 'parameters'):
    try:
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        report['trainable_ratio'] = trainable/total if total else None
    except Exception:
        pass
# ベンチ結果取り込み (最後の行平均)
import pandas as _pd
if BENCH_CSV.exists():
    dfb = _pd.read_csv(BENCH_CSV)
    if len(dfb):
        report['bench_avg_tok_per_s'] = float(dfb['tok_per_s'].replace('',0).astype(float).mean())
with open(RUN_DIR/"report.json","w",encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(report)


In [ ]:
# 23. バージョニング管理 (latest シンボリックリンク)
import os, pathlib
latest_link = OUTPUT_BASE / 'latest'
try:
    if latest_link.exists() or latest_link.is_symlink():
        if latest_link.is_dir() and not latest_link.is_symlink():
            pass
        latest_link.unlink()
    os.symlink(RUN_DIR, latest_link, target_is_directory=True)
    print("[SYMLINK] latest ->", RUN_DIR)
except Exception as e:
    print("[INFO] symlink skipped:", e)


In [ ]:
# 24. VRAM / メモリ解放ユーティリティ
import gc

def free_vram(var_names=None):
    if var_names is None:
        var_names = ['train_tok','val_tok','train_ds','val_ds','model','tokenizer','collator']
    for n in var_names:
        if n in globals():
            obj = globals()[n]
            try:
                if isinstance(obj, torch.nn.Module):
                    obj.to('cpu')
            except Exception:
                pass
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    print("[FREE] done")

# free_vram()  # 必要なときに実行


# 25. 追加改善アイデア

- AWQ / GPTQ / EXL2 など事後量子化を merged モデルに適用して比較 (速度/VRAM/品質)。
- 追加評価指標: RougeL, BLEU, BERTScore, hallucination proxy (正規表現で禁止語チェック等)。
- 長文 (8k–32k) コンテキストでのスループット/メモリ計測セル拡張。
- 温度/Top-p スイープ自動化 → JSONL に各生成とメトリクスを保存。
- LoRA Rank スイープ (r ∈ {4,8,16,32}) ベンチ自動化。
- 早期停止 (EarlyStoppingCallback) 導入による学習時間短縮。
- Repro: config_hash をベンチ & レポート両CSVに必ず記録し差分検知スクリプト追加。


In [ ]:
# (追加) 21b. 連続対話ループ (ユーザー入力で継続)
import json, datetime

def chat_loop(stop_words=None, max_turns=None, save_path=None):
    """
    stop_words: 終了トリガ語 (set)
    max_turns: 上限ターン数 (None なら無制限)
    save_path: 履歴を JSON 保存するパス (None で保存しない)
    """
    if stop_words is None:
        stop_words = {"exit","quit",":q","/exit"}
    turn = 0
    print(f"[Chat Loop 開始] 終了ワード: {stop_words}")
    while True:
        if max_turns is not None and turn >= max_turns:
            print("[INFO] max_turns 到達で終了")
            break
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[中断]")
            break
        if user_text.lower() in stop_words:
            print("[終了トリガ受信]")
            break
        if not user_text:
            continue
        # 1ターン応答
        history.append({'role':'user','content':user_text})
        prompt = build_prompt(history)
        enc = tokenizer(prompt, return_tensors='pt')
        if use_cuda:
            enc = {k:v.to('cuda') for k,v in enc.items()}
        print('Assistant:', end=' ', flush=True)
        streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        with torch.inference_mode():
            model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, streamer=streamer,
                           eos_token_id=stop_ids, pad_token_id=tokenizer.pad_token_id)
        # 最後の差分抽出 (簡易: 全履歴からは再構築せず placeholder)
        turn += 1
    if save_path:
        try:
            with open(save_path,'w',encoding='utf-8') as f:
                json.dump({'created': datetime.datetime.utcnow().isoformat(), 'history': history}, f, ensure_ascii=False, indent=2)
            print('[SAVED]', save_path)
        except Exception as e:
            print('[WARN] save failed:', e)
    print('[Chat Loop 終了] 総ターン:', turn)

# 実行例 (必要なときだけアンコメント)
# chat_loop(save_path=str(RUN_DIR/"chat_history.json"))


In [ ]:
# (追加) 21c. 連続対話ベンチ付きループ
import time, csv as _csv

def chat_loop_bench(stop_words=None, max_turns=None, bench_csv=None):
    """連続対話を行い、各ターンでTTFT, total, tokens/secなどを計測してCSVへ追記する。
    bench_csv: None の場合は RUN_DIR/"chat_bench.csv" を使用。
    既存 bench.csv とは列構成を合わせ (一部専用列追加: user_len)
    """
    if stop_words is None:
        stop_words = {"exit","quit",":q","/exit"}
    if bench_csv is None:
        bench_csv = RUN_DIR/"chat_bench.csv"

    header = ["timestamp","config_hash","model_id","mode","user_len","prompt_tokens","new_tokens","ttft_ms","gen_ms","total_ms","tok_per_s","peak_mem_gb"]
    exists = bench_csv.exists()

    print(f"[ChatBench 開始] 終了ワード: {stop_words} 保存先: {bench_csv}")
    turn = 0
    while True:
        if max_turns is not None and turn >= max_turns:
            print("[INFO] max_turns 到達で終了")
            break
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[中断]")
            break
        if user_text.lower() in stop_words:
            print("[終了トリガ受信]")
            break
        if not user_text:
            continue
        history.append({'role':'user','content':user_text})
        prompt = build_prompt(history)
        enc = tokenizer(prompt, return_tensors='pt')
        if use_cuda:
            enc = {k:v.to('cuda') for k,v in enc.items()}
        if use_cuda:
            torch.cuda.reset_peak_memory_stats()
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        gen_kwargs = dict(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, streamer=streamer,
                          eos_token_id=stop_ids, pad_token_id=tokenizer.pad_token_id)
        start = time.time(); first_token_t=None; gen_tokens=0
        def _g():
            with torch.inference_mode():
                model.generate(**gen_kwargs)
        import threading
        th = threading.Thread(target=_g); th.start()
        for chunk in streamer:
            if first_token_t is None:
                first_token_t = time.time()
            gen_tokens += len(tokenizer(chunk).input_ids)
        th.join(); end = time.time()
        ttft_ms = (first_token_t-start)*1000 if first_token_t else None
        total_ms = (end-start)*1000
        gen_ms = (end-first_token_t)*1000 if first_token_t else None
        tok_per_s = gen_tokens/(end-first_token_t) if first_token_t else None
        peak_gb = torch.cuda.max_memory_allocated()/1024**3 if use_cuda else None
        row = [datetime.datetime.utcnow().isoformat(), config_hash, MODEL_ID, chosen_mode, len(user_text),
               enc['input_ids'].shape[-1], gen_tokens,
               round(ttft_ms,2) if ttft_ms else '', round(gen_ms,2) if gen_ms else '', round(total_ms,2),
               round(tok_per_s,2) if tok_per_s else '', round(peak_gb,3) if peak_gb else '']
        with open(bench_csv,'a',newline='') as f:
            w=_csv.writer(f)
            if not exists:
                w.writerow(header); exists=True
            w.writerow(row)
        print(f"[TURN {turn}] ttft={row[7]}ms total={row[9]}ms tok/s={row[10]}")
        turn += 1
    print('[ChatBench 終了] 総ターン:', turn)

# 実行例 (必要時アンコメント)
# chat_loop_bench(max_turns=5)
